# detfuse — Detector 評估

比較 **KeywordDetector**（L1 軟分數）、**Qwen2.5-0.5B**（L2 信心分數）與**並行融合**（α × L1 + (1-α) × L2）在偵測「免費食物」貼文的準確率。

執行環境：Colab（T4 GPU）

## 1. 安裝依賴

In [1]:
!pip install -q transformers accelerate peft trl "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.6 MB/s eta 0:00:00


## 2. 設定（BRANCH / SEED）

In [2]:
import random, os
import numpy as np
import torch
from transformers import set_seed

BRANCH = 'feature/experiment-05-seed-finetune'  # 必須與當前工作分支一致
SEED   = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f'BRANCH={BRANCH}  SEED={SEED}')

BRANCH=feature/experiment-05-seed-finetune  SEED=42


## 3. KeywordDetector（直接複製 detector.py 的 L1 邏輯）

In [3]:
import re

# 與 detector.py 保持一致
_FREE_WORDS = r'免費|free|請拿|拿走|多餘|多的|送人|不要了|剩食|剩菜|拿去|有需要|帶走|送出|分享'
_FOOD_WORDS = r'食物|食品|飯|麵|便當|零食|餅乾|水果|蔬菜|菜|湯|肉|蛋|麵包|吐司|料理|點心|糕|餅|粽|飲料|奶茶|咖啡|茶|寶特瓶|三明治|沙拉|漢堡|披薩|壽司|飯糰|泡麵|湯圓'
_PATTERN_FREE_FOOD    = re.compile(rf'(?=.*({_FREE_WORDS}))(?=.*({_FOOD_WORDS}))', re.IGNORECASE)
_PATTERN_NOT_FOOD     = re.compile(r'免費.*?(?:課程|諮詢|活動|講座|workshop|票|名額|參加|索取)', re.IGNORECASE)
_PATTERN_EVENT_FOOD   = re.compile(
    r'(?:研討會|活動|工作坊|演講|說明會|工作人員).{0,10}(?:便當|餐盒|餐點|飲料|食物|點心)',
    re.IGNORECASE,
)
_PATTERN_HAS_CATERING = re.compile(r'(?:有|免費|提供)供餐', re.IGNORECASE)


def keyword_detect(text: str) -> bool:
    if not text:
        return False
    if _PATTERN_NOT_FOOD.search(text):
        return False
    return bool(_PATTERN_FREE_FOOD.search(text))


def l1_score(text: str) -> float:
    """L1 軟分數，[0, 1]。"""
    score = 0.0
    if _PATTERN_NOT_FOOD.search(text):
        return 0.0
    if _PATTERN_FREE_FOOD.search(text):     score += 0.80
    if _PATTERN_EVENT_FOOD.search(text):    score += 0.60
    if _PATTERN_HAS_CATERING.search(text):  score += 0.50
    has_free = bool(re.search(_FREE_WORDS, text, re.IGNORECASE))
    has_food = bool(re.search(_FOOD_WORDS, text, re.IGNORECASE))
    if has_free and not _PATTERN_FREE_FOOD.search(text): score += 0.20
    if has_food and not _PATTERN_FREE_FOOD.search(text): score += 0.10
    return min(score, 1.0)


print('L1 patterns + l1_score() loaded')

L1 patterns + l1_score() loaded


## 4. 測試資料集

**標記說明**：`1` = 確實在送免費食物，`0` = 非食物或不是免費送

> 上線前請補充真實社團貼文（正例 + 負例各 ≥ 15 篇）

In [4]:
SAMPLES = [
    # ── 正例（label=1）真實社團貼文 ──────────────────────────────────────────────
    # #18 研討會飲料剩餘
    (1, '不好意思又打擾了～\n研討會小強不夠多所以飲料還有剩，歡迎大家來喝～\n地點：航太系館1樓\n種類：冬瓜停檸、無糖四季春青茶、黑糖紅茶'),
    # #20 系統系館研討會多餘便當
    (1, '在系統系館中庭後面的長桌有研討會多出來的便當~~\n要取要快 食安自負'),
    # #36 活動多的披薩（NT$67 = 免費梗，現代年輕人用法）
    (1, '活動多的披薩，食安靖自行負真\n地點：系統系館\n時間：21:00前'),
    # #40 研討會便當
    (1, '研討會多的便當，超級好吃，歡迎前來自取\n地點：航太系館1樓'),
    # #43 典型正例：食物照片 + 免費標題 + 地點 + 食安自負
    (1, '研討會多的便當，超級好吃，歡迎前來自取\n地點：航太系館1樓\n食安自負'),
    # #45 舞展工作人員有免費供餐（付出勞力但有免費食物）
    (1, '成大舞研年度舞展小黑人招募中！\n福利：當日免費供餐！可提早於最佳觀眾席看到舞者們精彩的演出！\n日期：6/27（舞展當日）\n服裝：全身黑色（工作基本dress code）'),
    # #46 餅乾不合口味故送出
    (1, '上禮拜去農會超市買的餅乾 不合口味\n故送出\n青心烏龍茶燒米餅 有七包'),
    # #47 研討會多的日式便當
    (1, '研討會多的日式便當，目前有10個\n太子文旅三樓自取'),
    # #48 贈送貓咪零食餐包（寵物食品亦算）
    (1, '（已預訂）送全新未折貓咪零食跟餐包\n效期都在2027之後（藍莓小貓頭是2026九月）\n家裡的挑食怪連零食都挑 受不了\n送給有需要的人，東平路烏麵包附近自取～'),
    # #49/#50 Meetup 活動有供餐
    (1, '5/12(二)Meetup活動資訊：\n主題：智慧機器人產業應用趨勢\n時間：5/12（二）18:00 ～ 20:00（17:45開始到場）\n地點：陽明交通大學台南校區 奇美樓218教室\n注意事項：有供餐'),
    # ── 負例（label=0）真實社團貼文 ──────────────────────────────────────────────
    # #15 心理量表問卷招募（免費活動，非食物）
    (0, '大家好！我們是成心理系修習【量表編製與評估】課程的學生，目前正在進行「純威力量表」的編製研究，想了解大學生在面對壓力、批評與情緒波動時的反應方式與心理調適傾向。誠摯邀請大家協助填寫問卷囉'),
    # #16 跨感官知覺研究招募
    (0, '各位社團朋友好，我是成大心理系黃君群老師實驗室研究負責人 張良聖。目前實驗室正在進行一項短期學術研究，邀請大家在電腦前勤動手指，協助我們完成線上調查！研究主題：跨感官體驗與美感知覺（味覺/形狀聯結、書法偏好評估）'),
    # #17 化學書鍵盤販售
    (0, '賣舊書賣鍵盤，有些附小贈品，橫批絕處逢生\n微積分、普物、物化、無機、分析化學教科書，統一價800\n化學系本本 -100，被當過主科 -100\n狼蛛 F75雪杉綠鍵盤 700'),
    # #19 南山公墓田野調查問卷
    (0, '大家午安，我們是來自成大台文系的學生，有一門必修小專題，我們是做關於南山公墓與都市更新的研究，急需田野調查資料，希望同學能夠撥冗填寫。填寫時間大約三分鐘！'),
    # #21 免費贈送小玩具（非食物）
    (0, '長樂路二段全聯或南紡購物中心面交\n【免費贈送】一些不玩的小玩具'),
    # #26 畢業大拍賣（FB 亂填免費，其實是有價販售）
    (0, '即將畢業搬家出清！生活好物便宜到愛，數量有限，售完為止！\n大同100L單門小冰箱、宜得利矮立間坐面和坐椅、喜菲久坐不鏽腰坐墊、木紋實用摺疊桌、Apple Pencil 1、HP H100電競耳機、SAMPO聲寶捕蚊燈\n台南市區or成大自強校區面交'),
    # #27 一番賣出清（FB 亂填免費，其實是有價販售）
    (0, '一番賣出清～這次被貓巨破錢包了😅\n價格都在圖上 有問路都歡迎詢問\n多收優先\n南台面交or賣貨便須匯款就+$20\n行李箱只接受面交'),
    # #28 車禍徵目擊者
    (0, '在5/26約早上9點50分時，在長樂路四段5號（新K對面）發生一場車禍，但我的車上沒有安裝行車記錄器，想詢問大家是否有人經過可以提供畫面；我會再請你喝星巴克的'),
    # #29/#30 生育意願問卷
    (0, '大家好，我們是114-2社會心理學的修課學生，目前正在進行課堂報告關於「對於生育意願的態度」的研究調查。填答條件：具中華民國國籍與華文閱讀能力者皆可填答。本問卷不具名，填答時間約3-5分鐘，沒有標準答案'),
    # #31 搬家出清（有標價，已預訂/吊出狀態）
    (0, '即將畢業搬家出清，生活好物便宜到愛，數量有限，售完為止！\n大同100L單門小冰箱 (TR-100S) (已預訂)、宜得利矮立間坐面和坐椅 (已吊出)、喜菲久坐不鏽腰坐墊 (已吊出)、木紋實用摺疊桌 (已吊出)\nApple Pencil 1、HP H100電競耳機、SAMPO聲寶捕蚊燈、雷達薄型液體電蚊香'),
    # #34 雜物娃娃出清（有標價）
    (0, '（暫售）免打孔的桌面上加寬延伸板：50\n排球少年赤葦京治公仔 全新：600\n（暫售）1.8L帶蒸籠萬用鍋 買來一年沒用過：400\nHAPIINS奶酪貓 全新：150\nHAPIINS牛奶貓 全新：150\n超大兔卡提西亞娃娃：500\n黑糖鮮奶麻糬湯材料組合包（全新未開封）：50'),
    # #35 搬家出清 IKEA（有標價）
    (0, '搬家出清！！！ 價格如圖～～～🙏\nIKEA娃娃兩隻一起$400'),
    # #37 協尋鑰匙（失物招領）
    (0, '地點：成大自強校區儀器設備大樓9樓\n近期有到儀器設備大樓B1無塵室做實驗的同學，你的鑰匙掉在無塵衣的口袋裡，被我們撿到，看到貼文的話，請盡快來9樓核心設施中心櫃台認領。'),
    # #38/#39 成心城涌錯覺展演（免費活動，非食物）
    (0, '成心城涌，河以路尋：第三屆成大心理系錯覺展演\n你看到的，真的是你看到的嗎？今年夏天，成功大學心理系與成功大學歷史系及日本立命館大學合作籌辦\n展覽資訊：5/21(四)－5/23(六) 9:00-21:00 @ 成大歷史文物館\n費用：免費'),
    # #41 南山公墓社遊（免費社遊，非食物）
    (0, '（社遊借版宣傳）\n延續上次社社的主題，目前位於台南機場北側，擁有超過400年歷史的南山公墓，目前正面臨被拆除、強制遷葬的危機。本次湯社的社遊將由南山國家墓葬歷史生態區促進會秘書長擔任導覽員\n費用：免費\n時間：5/24（日）14:00-15:30'),
    # #42 徵二手電風扇（徵收，非免費送）
    (0, '大家好，我想買一台二手電風扇。如果有人想轉讓的話，請留言或私訊我價格和狀況。謝謝！'),
    # #44 搬家出清（有標價）
    (0, '搬家出清！！！ 價格如圖～～～🙏\nIKEA娃娃兩隻$400\n各式碗盤 $15/個，共五個全部走$60'),
]

print(f'資料集：{sum(l==1 for l,_ in SAMPLES)} 正例 / {sum(l==0 for l,_ in SAMPLES)} 負例')

資料集：10 正例 / 17 負例


In [5]:
import json, urllib.request

_BASE = (
    f'https://raw.githubusercontent.com/syoslyot/detfuse/{BRANCH}'
    '/data/categories/free_food'
)

def _load_json(filename):
    url = f'{_BASE}/{filename}'
    with urllib.request.urlopen(url) as r:
        data = json.load(r)
    return [(d['label'], d['text']) for d in data]

TRAIN_SAMPLES = _load_json('training_data.json')
TEST_SAMPLES  = _load_json('test_data.json')

print(f'TRAIN: {sum(l==1 for l,_ in TRAIN_SAMPLES)} pos / {sum(l==0 for l,_ in TRAIN_SAMPLES)} neg')
print(f'TEST:  {sum(l==1 for l,_ in TEST_SAMPLES)} pos / {sum(l==0 for l,_ in TEST_SAMPLES)} neg')

TRAIN: 215 pos / 679 neg
TEST:  67 pos / 167 neg


## 5. 評估 KeywordDetector

In [6]:
def evaluate(name, predict_fn, samples):
    tp = fp = tn = fn = 0
    errors = []
    for label, text in samples:
        pred = predict_fn(text)
        if label == 1 and pred:     tp += 1
        elif label == 0 and not pred: tn += 1
        elif label == 0 and pred:
            fp += 1
            errors.append(('FP', text[:60]))
        else:
            fn += 1
            errors.append(('FN', text[:60]))

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall    = tp / (tp + fn) if (tp + fn) else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    print(f'\n── {name} ──')
    print(f'  Precision: {precision:.2%}  Recall: {recall:.2%}  F1: {f1:.2%}')
    print(f'  TP={tp} FP={fp} TN={tn} FN={fn}')
    if errors:
        print('  錯誤案例:')
        for tag, t in errors:
            print(f'    [{tag}] {t}')
    return dict(name=name, precision=precision, recall=recall, f1=f1)

kw_result = evaluate('KeywordDetector (L1)', keyword_detect, TEST_SAMPLES)


── KeywordDetector (L1) ──
  Precision: 85.37%  Recall: 52.24%  F1: 64.81%
  TP=35 FP=6 TN=161 FN=32
  錯誤案例:
    [FN] 更 沒了
研討會剩下的便當
地點：國際會議廳多功能廳
食安自負，送完為止
更 沒了
研討會剩下的便當
地點：國際會議廳多
    [FN] 更 又拿完了 來不及更新
營隊吃不完的午餐 食安自負
醬自取可盡情取
老地方 社科院北棟二樓樓梯口
麻煩最後一位留個言感
    [FN] 研討會哈蜜瓜便當
座標成大光復校區中文系館演講廳（大門口）
研討會哈蜜瓜便當
座標成大光復校區中文系館演講廳（大門口）

    [FN] (發完囉!)
研討會便當-社科院南棟一樓
有需要的人歡迎來領取（食安自負）
    [FN] 營隊剩下的熏雞吐司
在社科院80103 食安自負
麻煩最後一個拿完的人留言

陳證仰
營隊剩下的熏雞吐司
在社科院801
    [FN] #更 已發完謝謝大家
外文研討會多出來的餐盒
讓你早餐吃飽飽～
放在成功湖邊（修齊大樓門口）的石桌上
歡迎自取
    [FN] 研討會七飯亭便當34個
座標成大光復校區中文系演講廳（大門口）
食安自負
研討會七飯亭便當34個
座標成大光復校區中文系
    [FN] 系統系研討會免費餐盒（已發完）
拿幾個都可以，先拿先贏！
系統系研討會免費餐盒（已發完）
拿幾個都可以，先拿先贏！
免費
    [FN] 資源系研討會好吃的午餐，有素食便當。
要自備餐具
在舊系館進來後右轉，到2:20要來要快。
資源系研討會好吃的午餐，有素
    [FN] 中午活動剩下的便當跟飲料（需自備環保杯），在光復校區國際會議廳請自取，到16:45，食安自負（都在冷氣房裡）。
免費
 
    [FN] 更 沒了
研討會剩下的便當
地點：國際會議廳多功能廳
食安自負，送完為止
更 沒了
研討會剩下的便當
地點：國際會議廳多
    [FN] 超級好吃便當還有三分春色的飲料、啊品項很多啦自己挑，放在藝研所東側門，食安自負！
$1
 

（已售出） 來吃飯

（已
    [FN] 活動剩下很多
 熱咖啡 
 熱奶茶 
 梅子綠 
熱麥茶
快來哦！歡迎裝爆
在國際會議廳 一活多功能廳這裡！
我們也會去
    [FN]

## 6. Qwen2.5-0.5B-Instruct（L2 模型，需 GPU）

模擬 `OllamaDetector` 的邏輯，但改用 HuggingFace Transformers 直接跑。
這樣不需要 Ollama server，在 Colab 上也能驗證模型品質。

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)
print(f'模型載入完成，裝置：{next(model.parameters()).device}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

模型載入完成，裝置：cuda:0


In [8]:
PROMPT_TMPL = (
    '判斷以下貼文是否在提供免費食物或飲料（可以現在就去拿）。\n'
    '請只輸出 0 到 9 的整數，代表信心程度（0 = 完全不是，9 = 完全確定是）。不要輸出其他任何文字。\n\n'
    '貼文：{text}'
)


def qwen_prob(text: str) -> float:
    """回傳 [0, 1] 信心分數（0-9 digit / 9.0）。"""
    messages = [
        {'role': 'system', 'content': '請只輸出 0 到 9 的整數。'},
        {'role': 'user', 'content': PROMPT_TMPL.format(text=text[:400])},
    ]
    ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    )
    if hasattr(ids, 'input_ids'):
        ids = ids.input_ids
    ids = ids.to(model.device)
    input_len = ids.shape[1]
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=5, do_sample=False)
    answer = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
    for ch in answer:
        if ch.isdigit():
            return int(ch) / 9.0
    return 0.5  # 無法解析 → 中立


def qwen_detect(text: str) -> bool:
    return qwen_prob(text) > 0.5


# Smoke test
print(qwen_prob('有多的便當，免費拿走，在工程館'))   # expect ≥ 0.5
print(qwen_prob('出售二手書'))                        # expect < 0.5

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


0.7777777777777778
0.1111111111111111


In [9]:
qwen_result = evaluate('Qwen2.5-0.5B (L2)', qwen_detect, TEST_SAMPLES)


── Qwen2.5-0.5B (L2) ──
  Precision: 16.48%  Recall: 22.39%  F1: 18.99%
  TP=15 FP=76 TN=91 FN=52
  錯誤案例:
    [FN] 《免費便當、沙拉、湯》（沒了）
系展剩下的，歡迎來成功校區地科系館1F自取
可以順便來看地科系展
《免費便當、沙拉、湯》
    [FP] 寒假的營隊在光餐的剩食
讓阿姨見識到二手版的威力
所以認識的阿姨請我幫忙宣導
光復自助餐的半價時段
星期一~四 18：3
    [FP] 更/拿完了謝謝各位
午安您好，又到了午飯時間。
/一樣建築系館旁的遮陽地方/
今日菜單
刈包+碗粿*4
刈包+米糕*5

    [FP] 【出售】大同 TATUNG 51L 經典小冰箱
• 商品名稱： 大同冷藏電冰箱 (TR-50HC)
• 商品尺寸： 寬 
    [FP] 是誰把我的帥氣老婆弄倒了⋯
下午騎車發現左邊煞車凹損
車體左側有多處擦傷
5/16（六）
8:40-16:10 停放於雲
    [FN] 光復校區-國際會議廳一樓多功能廳
更新 拿完了
感恩的心
光復校區-國際會議廳一樓多功能廳
更新 拿完了
感恩的心
免費
    [FP] 出售未使用到的商品～
1. One meter輕巧負離子高速吹風機 $1500（全新）- 原價2980
2. KD203
    [FP] 徵好心學弟妹幫我買香格里拉杜拜巧克力Q餅到Costco附近
跑腿費250
+一杯飲料實報實銷
時間是這週五下午
    [FN] 更 沒了
研討會剩下的便當
地點：國際會議廳多功能廳
食安自負，送完為止
更 沒了
研討會剩下的便當
地點：國際會議廳多
    [FP] ［預定中］
大四畢業出清！
【GIANT momentum 腳踏車 售$4000】
附贈：車燈+鎖
大二時於捷安特門市購
    [FN] 更 又拿完了 來不及更新
營隊吃不完的午餐 食安自負
醬自取可盡情取
老地方 社科院北棟二樓樓梯口
麻煩最後一位留個言感
    [FP] 更：找到了謝謝好心人qqqqqqqqqq
抱歉打擾
11/26 12:00左右 （中午時段）
走去醫學院的路上錢包不見了
    [FN] 營隊有剩餘便當 放在軍訓室前的桌子上
#食安自負
營隊有剩餘便當 放在軍訓室前的桌子上
#食安自負

## 7. 並行融合（α × L1 + (1-α) × L2）

L1 和 L2 各自計算分數，再加權融合——兩層都有發言權，都能糾正對方的誤判。

In [10]:
def fuse(text: str, alpha: float = 0.35, threshold: float = 0.50) -> bool:
    s1 = l1_score(text)
    s2 = qwen_prob(text)
    return (alpha * s1 + (1 - alpha) * s2) > threshold


fusion_result = evaluate('Fusion α=0.35 τ=0.50', fuse, TEST_SAMPLES)


── Fusion α=0.35 τ=0.50 ──
  Precision: 21.88%  Recall: 31.34%  F1: 25.77%
  TP=21 FP=75 TN=92 FN=46
  錯誤案例:
    [FN] 《免費便當、沙拉、湯》（沒了）
系展剩下的，歡迎來成功校區地科系館1F自取
可以順便來看地科系展
《免費便當、沙拉、湯》
    [FP] 寒假的營隊在光餐的剩食
讓阿姨見識到二手版的威力
所以認識的阿姨請我幫忙宣導
光復自助餐的半價時段
星期一~四 18：3
    [FP] 更/拿完了謝謝各位
午安您好，又到了午飯時間。
/一樣建築系館旁的遮陽地方/
今日菜單
刈包+碗粿*4
刈包+米糕*5

    [FP] 【出售】大同 TATUNG 51L 經典小冰箱
• 商品名稱： 大同冷藏電冰箱 (TR-50HC)
• 商品尺寸： 寬 
    [FP] 是誰把我的帥氣老婆弄倒了⋯
下午騎車發現左邊煞車凹損
車體左側有多處擦傷
5/16（六）
8:40-16:10 停放於雲
    [FN] 光復校區-國際會議廳一樓多功能廳
更新 拿完了
感恩的心
光復校區-國際會議廳一樓多功能廳
更新 拿完了
感恩的心
免費
    [FP] 出售未使用到的商品～
1. One meter輕巧負離子高速吹風機 $1500（全新）- 原價2980
2. KD203
    [FP] 徵好心學弟妹幫我買香格里拉杜拜巧克力Q餅到Costco附近
跑腿費250
+一杯飲料實報實銷
時間是這週五下午
    [FN] 更 沒了
研討會剩下的便當
地點：國際會議廳多功能廳
食安自負，送完為止
更 沒了
研討會剩下的便當
地點：國際會議廳多
    [FP] ［預定中］
大四畢業出清！
【GIANT momentum 腳踏車 售$4000】
附贈：車燈+鎖
大二時於捷安特門市購
    [FN] 更 又拿完了 來不及更新
營隊吃不完的午餐 食安自負
醬自取可盡情取
老地方 社科院北棟二樓樓梯口
麻煩最後一位留個言感
    [FP] 更：找到了謝謝好心人qqqqqqqqqq
抱歉打擾
11/26 12:00左右 （中午時段）
走去醫學院的路上錢包不見了
    [FN] 營隊有剩餘便當 放在軍訓室前的桌子上
#食安自負
營隊有剩餘便當 放在軍訓室前的桌子上
#食

## 8. 比較結果

In [11]:
print(f'\n{"模型":<28} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 62)
for r in [kw_result, qwen_result, fusion_result]:
    print(f"{r['name']:<28} {r['precision']:>10.2%} {r['recall']:>10.2%} {r['f1']:>10.2%}")


模型                            Precision     Recall         F1
--------------------------------------------------------------
KeywordDetector (L1)             85.37%     52.24%     64.81%
Qwen2.5-0.5B (L2)                16.48%     22.39%     18.99%
Fusion α=0.35 τ=0.50             21.88%     31.34%     25.77%


## 9. Fine-tuning（LoRA）

In [12]:
from peft import get_peft_model, LoraConfig, TaskType
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup

# ── SFTDataset：只對 assistant 回覆 token 計算 loss ──────────────────────
class SFTDataset(torch.utils.data.Dataset):
    def __init__(self, samples, tokenizer, max_length=512):
        self.items = []
        for label, text in samples:
            messages = [
                {'role': 'system', 'content': '請只輸出 0 到 9 的整數。'},
                {'role': 'user', 'content': PROMPT_TMPL.format(text=text[:400])},
            ]
            prompt_text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
            answer_ids = tokenizer.encode('9' if label == 1 else '0', add_special_tokens=False)
            eos_id     = [tokenizer.eos_token_id]
            input_ids  = prompt_ids + answer_ids + eos_id
            prompt_len = len(prompt_ids)
            labels     = [-100] * prompt_len + answer_ids + eos_id
            if len(input_ids) > max_length:
                continue
            self.items.append({
                'input_ids': torch.tensor(input_ids),
                'labels':    torch.tensor(labels),
            })

    def __len__(self):           return len(self.items)
    def __getitem__(self, idx):  return self.items[idx]


def collate_fn(batch):
    pad_id = tokenizer.pad_token_id
    input_ids = torch.nn.utils.rnn.pad_sequence(
        [b['input_ids'] for b in batch], batch_first=True, padding_value=pad_id)
    labels = torch.nn.utils.rnn.pad_sequence(
        [b['labels'] for b in batch], batch_first=True, padding_value=-100)
    attention_mask = (input_ids != pad_id).long()
    return {'input_ids': input_ids, 'labels': labels, 'attention_mask': attention_mask}


dataset = SFTDataset(TRAIN_SAMPLES, tokenizer)
# 驗證 loss masking
sample = dataset[0]
total_tok = len(sample['input_ids'])
label_tok = sum(1 for l in sample['labels'].tolist() if l != -100)
print(f'Total tokens: {total_tok}, Label tokens (assistant only): {label_tok}')

Total tokens: 253, Label tokens (assistant only): 2


In [13]:
# ── LoRA 設定 ─────────────────────────────────────────────────────────────
lora_cfg = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
ft_model = get_peft_model(model, lora_cfg)
ft_model.print_trainable_parameters()

# ── 訓練 ──────────────────────────────────────────────────────────────────
EPOCHS     = 3
BATCH_SIZE = 4
LR         = 2e-4

loader    = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
optimizer = torch.optim.AdamW(ft_model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=len(loader) * EPOCHS,
)

ft_model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch in loader:
        batch = {k: v.to(ft_model.device) for k, v in batch.items()}
        loss  = ft_model(**batch).loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}: avg loss = {total_loss / len(loader):.4f}')

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184
Epoch 1: avg loss = 0.1402
Epoch 2: avg loss = 0.0515
Epoch 3: avg loss = 0.0161


In [14]:
# ── 評估 fine-tuned 模型 ──────────────────────────────────────────────────
ft_model.eval()

def qwen_ft_prob(text: str) -> float:
    messages = [
        {'role': 'system', 'content': '請只輸出 0 到 9 的整數。'},
        {'role': 'user', 'content': PROMPT_TMPL.format(text=text[:400])},
    ]
    ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt')
    if hasattr(ids, 'input_ids'):
        ids = ids.input_ids
    ids = ids.to(ft_model.device)
    input_len = ids.shape[1]
    with torch.no_grad():
        out = ft_model.generate(ids, max_new_tokens=5, do_sample=False)
    answer = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
    for ch in answer:
        if ch.isdigit():
            return int(ch) / 9.0
    return 0.5

def qwen_ft_detect(text: str) -> bool:
    return qwen_ft_prob(text) > 0.5

ft_result = evaluate('Qwen2.5-0.5B fine-tuned (L2)', qwen_ft_detect, TEST_SAMPLES)


── Qwen2.5-0.5B fine-tuned (L2) ──
  Precision: 98.33%  Recall: 88.06%  F1: 92.91%
  TP=59 FP=1 TN=166 FN=8
  錯誤案例:
    [FN] 更 又拿完了 來不及更新
營隊吃不完的午餐 食安自負
醬自取可盡情取
老地方 社科院北棟二樓樓梯口
麻煩最後一位留個言感
    [FN] 《免費領取》
在學生會會辦有免費的飲料、濕紙巾可以領取喔！要來要快！！
只到18：00喔！
《免費領取》
在學生會會辦有
    [FN] 營隊剩下的熏雞吐司
在社科院80103 食安自負
麻煩最後一個拿完的人留言

陳證仰
營隊剩下的熏雞吐司
在社科院801
    [FN] 勝利校區颱風天不想出去買飯ㄉ可以到v8、9警衛室領取免費便當
有宮保雞丁、燒肉、打拋豬
數量有限送完為止！
    [FN] 研討會多的餐盒
內容物是米糕、肉圓、魚丸湯之類的台南小吃～
地點在中文系館！
需要的可以過來拿
各位趕緊
研討會多的餐盒
    [FN] 營隊剩下的大肥
在社科院80103 食安自負
打卡並不會送黑白切
顯示更多

陳證仰
營隊剩下的大肥
在社科院80103
    [FN] 研討會送幸福！
免費餐點、茶點（今天沒吃完要冰冰箱喔）
食安自負 
化工系館門口
速速
    [FP] 免費
 

（已售出） 社團博覽會｜成大社聯會ig追蹤換免費飲料

（已售出）  社團博覽會｜成大社聯會ig追蹤換免費飲
    [FN] 電機營多的9個便當，在雲平大樓。
一樣麻煩最後一個拿的留言，感謝！
電機營多的9個便當，在雲平大樓。
一樣麻煩最後一個拿


In [15]:
# ── Fusion with fine-tuned ────────────────────────────────────────────────
def fuse_ft(text: str, alpha: float = 0.35, threshold: float = 0.50) -> bool:
    return (alpha * l1_score(text) + (1 - alpha) * qwen_ft_prob(text)) > threshold

fusion_ft_result = evaluate('Fusion ft α=0.35 τ=0.50', fuse_ft, TEST_SAMPLES)


── Fusion ft α=0.35 τ=0.50 ──
  Precision: 98.33%  Recall: 88.06%  F1: 92.91%
  TP=59 FP=1 TN=166 FN=8
  錯誤案例:
    [FN] 更 又拿完了 來不及更新
營隊吃不完的午餐 食安自負
醬自取可盡情取
老地方 社科院北棟二樓樓梯口
麻煩最後一位留個言感
    [FN] 《免費領取》
在學生會會辦有免費的飲料、濕紙巾可以領取喔！要來要快！！
只到18：00喔！
《免費領取》
在學生會會辦有
    [FN] 營隊剩下的熏雞吐司
在社科院80103 食安自負
麻煩最後一個拿完的人留言

陳證仰
營隊剩下的熏雞吐司
在社科院801
    [FN] 勝利校區颱風天不想出去買飯ㄉ可以到v8、9警衛室領取免費便當
有宮保雞丁、燒肉、打拋豬
數量有限送完為止！
    [FN] 研討會多的餐盒
內容物是米糕、肉圓、魚丸湯之類的台南小吃～
地點在中文系館！
需要的可以過來拿
各位趕緊
研討會多的餐盒
    [FN] 營隊剩下的大肥
在社科院80103 食安自負
打卡並不會送黑白切
顯示更多

陳證仰
營隊剩下的大肥
在社科院80103
    [FN] 研討會送幸福！
免費餐點、茶點（今天沒吃完要冰冰箱喔）
食安自負 
化工系館門口
速速
    [FP] 免費
 

（已售出） 社團博覽會｜成大社聯會ig追蹤換免費飲料

（已售出）  社團博覽會｜成大社聯會ig追蹤換免費飲
    [FN] 電機營多的9個便當，在雲平大樓。
一樣麻煩最後一個拿的留言，感謝！
電機營多的9個便當，在雲平大樓。
一樣麻煩最後一個拿


In [16]:
# ── 完整結果彙總 ──────────────────────────────────────────────────────────
print(f'\n{"模型":<35} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 67)
for r in [kw_result, qwen_result, fusion_result, ft_result, fusion_ft_result]:
    print(f"{r['name']:<35} {r['precision']:>10.2%} {r['recall']:>10.2%} {r['f1']:>10.2%}")


模型                                   Precision     Recall         F1
-------------------------------------------------------------------
KeywordDetector (L1)                    85.37%     52.24%     64.81%
Qwen2.5-0.5B (L2)                       16.48%     22.39%     18.99%
Fusion α=0.35 τ=0.50                    21.88%     31.34%     25.77%
Qwen2.5-0.5B fine-tuned (L2)            98.33%     88.06%     92.91%
Fusion ft α=0.35 τ=0.50                 98.33%     88.06%     92.91%


## 10. 上傳 HuggingFace

In [17]:
HF_REPO        = 'syoslyot/qwen-detfuse-finetuned'
EXPERIMENT_TAG = f'experiment_05 (SEED={SEED})'

ft_model.push_to_hub(HF_REPO, commit_message=EXPERIMENT_TAG)
tokenizer.push_to_hub(HF_REPO)
print(f'上傳完成：{HF_REPO}  [{EXPERIMENT_TAG}]')

HfHubHTTPError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6a1c767b-538adef42e4c6982040d8c83;3ab8bdd6-d1af-4a72-8d34-02681a537aa8)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

## 11. 新增自訂測試樣本

把遇到的真實貼文貼進來，看哪層漏掉、原因為何。

In [ ]:
CUSTOM = [
    # 貼上真實貼文測試，格式：(true_label, '貼文內容')
    # (1, '...'),
    # (0, '...'),
]

if CUSTOM:
    print('=== KeywordDetector ===')
    evaluate('KeywordDetector', keyword_detect, CUSTOM)
    print('\n=== Qwen2.5 ===')
    evaluate('Qwen2.5', qwen_detect, CUSTOM)
    print('\n=== TwoLayer ===')
    evaluate('TwoLayer', two_layer_detect, CUSTOM)
else:
    print('CUSTOM 為空，請填入真實貼文後重新執行')

## 12. 診斷分析（零樣本模型）

分析 L1/L2 分歧、alpha sweep、Precision-Recall Curve。

In [ ]:
from sklearn.metrics import precision_recall_curve, auc
import matplotlib.pyplot as plt

def collect_scores(score_fn, samples):
    """回傳 (labels, scores) 供 PR curve 使用。"""
    labels, scores = [], []
    for label, text in samples:
        labels.append(label)
        scores.append(score_fn(text))
    return labels, scores

# ── 12.1 L1 vs Zero-shot L2 分歧樣本 ────────────────────────────────────
print('=== L1 與 zero-shot L2 分歧樣本 ===')
diverge_pre = []
for label, text in TEST_SAMPLES:
    s1 = l1_score(text)
    s2 = qwen_prob(text)
    if (s1 > 0.5) != (s2 > 0.5):
        diverge_pre.append((label, s1, s2, text))
        tag = '✓' if label == int(s2 > 0.5) else '✗'
        print(f'  {tag} label={label} L1={s1:.2f} L2={s2:.2f} | {text[:80]}')
print(f'\n共 {len(diverge_pre)} 筆分歧')
if len(diverge_pre) == 0:
    print('⚠ 分歧為 0：Fusion 永遠不改變 L2 的決策，L1 貢獻完全無效。')

In [ ]:
# ── 12.2 Alpha 掃描（zero-shot Fusion）────────────────────────────────
alphas = [i / 10 for i in range(11)]
zs_f1 = []
for a in alphas:
    r = evaluate(
        f'Fusion α={a:.1f}',
        lambda t, a=a: (a * l1_score(t) + (1 - a) * qwen_prob(t)) > 0.5,
        TEST_SAMPLES
    )
    zs_f1.append(r['f1'])

plt.figure(figsize=(8, 4))
plt.plot(alphas, zs_f1, marker='o', label='Fusion F1')
plt.axhline(kw_result['f1'], linestyle='--', color='steelblue',
            label=f"L1 alone ({kw_result['f1']:.2%})")
plt.axhline(qwen_result['f1'], linestyle='--', color='darkorange',
            label=f"L2 alone ({qwen_result['f1']:.2%})")
plt.xlabel('Alpha（L1 weight）')
plt.ylabel('F1 Score')
plt.title('Zero-shot Fusion F1 vs Alpha')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 12.3 Precision-Recall Curve（零樣本）──────────────────────────────
plt.figure(figsize=(8, 6))
for name, fn in [('KeywordDetector (L1)', l1_score),
                  ('Qwen2.5 zero-shot (L2)', qwen_prob)]:
    lbl, sc = collect_scores(fn, TEST_SAMPLES)
    p, r, _ = precision_recall_curve(lbl, sc)
    plt.plot(r, p, label=f'{name} (AUC-PR={auc(r, p):.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve（零樣本）')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Fine-tuned 模型診斷

L1/ft-L2 分歧分析、alpha sweep、PR curve、Sequential Detection。

In [ ]:
# ── 13.1 L1 vs Fine-tuned L2 分歧樣本 ─────────────────────────────────
print('=== L1 與 fine-tuned L2 分歧樣本 ===')
diverge_ft = []
for label, text in TEST_SAMPLES:
    s1 = l1_score(text)
    s2 = qwen_ft_prob(text)
    if (s1 > 0.5) != (s2 > 0.5):
        diverge_ft.append((label, s1, s2, text))
        tag = '✓' if label == int(s2 > 0.5) else '✗'
        print(f'  {tag} label={label} L1={s1:.2f} L2={s2:.2f} | {text[:80]}')
print(f'\n共 {len(diverge_ft)} 筆分歧')
if len(diverge_ft) == 0:
    print('⚠ 分歧為 0：fine-tuned L2 在所有樣本上與 L1 決策相同，'
          '\n  Fusion 永遠不改變結果——根因是 ft L2 輸出 0.0/1.0 極端值，'
          '\n  α × L1 擾動量不足以跨越 threshold=0.5。')

# ── 13.2 Alpha 掃描（Fine-tuned Fusion）────────────────────────────────
ft_f1s = []
for a in alphas:
    r = evaluate(
        f'Fusion ft α={a:.1f}',
        lambda t, a=a: (a * l1_score(t) + (1 - a) * qwen_ft_prob(t)) > 0.5,
        TEST_SAMPLES
    )
    ft_f1s.append(r['f1'])

plt.figure(figsize=(8, 4))
plt.plot(alphas, ft_f1s, marker='o', color='darkorange', label='Fusion ft F1')
plt.axhline(ft_result['f1'], linestyle='--', color='gray',
            label=f"ft L2 alone ({ft_result['f1']:.2%})")
plt.axhline(kw_result['f1'], linestyle='--', color='steelblue',
            label=f"L1 alone ({kw_result['f1']:.2%})")
plt.xlabel('Alpha（L1 weight）')
plt.ylabel('F1 Score')
plt.title('Fine-tuned Fusion F1 vs Alpha')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 13.3 Precision-Recall Curve（fine-tuned）────────────────────────────
plt.figure(figsize=(8, 6))
for name, fn in [('KeywordDetector (L1)', l1_score),
                  ('Qwen2.5 fine-tuned (L2)', qwen_ft_prob)]:
    lbl, sc = collect_scores(fn, TEST_SAMPLES)
    p, r, _ = precision_recall_curve(lbl, sc)
    plt.plot(r, p, label=f'{name} (AUC-PR={auc(r, p):.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve（fine-tuned）')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── 13.4 Sequential Detection（L1 先過，不確定才呼叫 L2）─────────────────
L1_HIGH = 0.7  # L1 確定為正
L1_LOW  = 0.3  # L1 確定為負
_l2_calls = [0]

def sequential_detect(text: str) -> bool:
    s1 = l1_score(text)
    if s1 >= L1_HIGH:
        return True
    if s1 <= L1_LOW:
        return False
    _l2_calls[0] += 1
    return qwen_ft_prob(text) > 0.5

seq_result = evaluate('Sequential (L1→ft L2)', sequential_detect, TEST_SAMPLES)
total = len(TEST_SAMPLES)
print(f'\nL2 呼叫次數：{_l2_calls[0]}/{total} ({_l2_calls[0]/total:.1%})')
print(f'節省 L2 呼叫：{total - _l2_calls[0]}/{total} ({(total - _l2_calls[0])/total:.1%})')

# ── 最終對比（含 Sequential）────────────────────────────────────────────
print(f'\n{"模型":<35} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 70)
for r in [kw_result, qwen_result, fusion_result, ft_result, fusion_ft_result, seq_result]:
    print(f"{r['name']:<35} {r['precision']:>10.2%} {r['recall']:>10.2%} {r['f1']:>10.2%}")